# Week 4: Building a Proper ML Pipeline with Feature Engineering

## 1. Setup & Data Loading

In [50]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.impute import SimpleImputer
import joblib

df = pd.read_csv('../data/telco_churn.csv')
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 50 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   str    
 1   Gender                             7043 non-null   str    
 2   Age                                7043 non-null   int64  
 3   Under 30                           7043 non-null   str    
 4   Senior Citizen                     7043 non-null   str    
 5   Married                            7043 non-null   str    
 6   Dependents                         7043 non-null   str    
 7   Number of Dependents               7043 non-null   int64  
 8   Country                            7043 non-null   str    
 9   State                              7043 non-null   str    
 10  City                               7043 non-null   str    
 11  Zip Code                           7043 non-null   int64  
 12  Lat

## 2. Initial Data Exploration & Leakage Check

Checking target distribution and identifying columns that leak target information (e.g., `Customer Status` directly mirrors `Churn Label`).

In [51]:
print(df['Churn Label'].value_counts())
print(df['Customer Status'].value_counts())
print(df[['Country', 'State', 'Quarter']].nunique())

Churn Label
No     5174
Yes    1869
Name: count, dtype: int64
Customer Status
Stayed     4720
Churned    1869
Joined      454
Name: count, dtype: int64
Country    1
State      1
Quarter    1
dtype: int64


## 3. Dropping Leaky and Zero-Variance Columns

Removing identifiers, leakage columns, and columns with only one unique value.

In [52]:
# Columns to drop: identifiers, leakage, zero-variance, redundant target-related
drop_cols = [
    'Customer ID', 'Country', 'State', 'Quarter',
    'Customer Status', 'Churn Score', 'Churn Category', 'Churn Reason', 'CLTV',
    'Lat Long' if 'Lat Long' in df.columns else None
]
drop_cols = [c for c in drop_cols if c is not None]

df_model = df.drop(columns=drop_cols)

# Separate target
y = df_model['Churn Label'].map({'Yes': 1, 'No': 0})
X = df_model.drop(columns=['Churn Label'])

print("Shape:", X.shape)
print(X.dtypes.value_counts())

Shape: (7043, 40)
str        23
int64       9
float64     8
Name: count, dtype: int64


## 4. Reviewing Column Types

In [53]:
print(X.select_dtypes(include=['int64', 'float64']).columns)
print()
print(X.select_dtypes(include=['str', 'object']).columns)

Index(['Age', 'Number of Dependents', 'Zip Code', 'Latitude', 'Longitude',
       'Population', 'Number of Referrals', 'Tenure in Months',
       'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download',
       'Monthly Charge', 'Total Charges', 'Total Refunds',
       'Total Extra Data Charges', 'Total Long Distance Charges',
       'Total Revenue', 'Satisfaction Score'],
      dtype='str')

Index(['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'City',
       'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines',
       'Internet Service', 'Internet Type', 'Online Security', 'Online Backup',
       'Device Protection Plan', 'Premium Tech Support', 'Streaming TV',
       'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract',
       'Paperless Billing', 'Payment Method'],
      dtype='str')


## 5. Checking High-Cardinality Columns

`City` and `Zip Code` have too many unique values to be usable in a one-hot encoded model.

In [54]:
print("City unique values:", X['City'].nunique())
print("Zip Code unique values:", X['Zip Code'].nunique())

City unique values: 1106
Zip Code unique values: 1626


## 6. Finalizing the Broad Feature Set

This was exploratory — it shows the full dataset after cleanup. In the next step, the feature set is narrowed down to match Week 3's manual approach, so pipeline results can be validly compared against it.

In [55]:
X = X.drop(columns=['City', 'Zip Code'])

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['str', 'object']).columns

print("Numerical columns:", len(numerical_cols))
print(numerical_cols)
print()
print("Categorical columns:", len(categorical_cols))
print(categorical_cols)

Numerical columns: 16
Index(['Age', 'Number of Dependents', 'Latitude', 'Longitude', 'Population',
       'Number of Referrals', 'Tenure in Months',
       'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download',
       'Monthly Charge', 'Total Charges', 'Total Refunds',
       'Total Extra Data Charges', 'Total Long Distance Charges',
       'Total Revenue', 'Satisfaction Score'],
      dtype='str')

Categorical columns: 22
Index(['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents',
       'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines',
       'Internet Service', 'Internet Type', 'Online Security', 'Online Backup',
       'Device Protection Plan', 'Premium Tech Support', 'Streaming TV',
       'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract',
       'Paperless Billing', 'Payment Method'],
      dtype='str')


## 7. Feature Selection — Scoped to Week 3

Using the same 6 features as the Week 3 manual model (`Tenure in Months`, `Monthly Charge`, `Contract`, `Payment Method`, `Internet Type`, `Total Charges`) so pipeline results are a fair, apples-to-apples comparison.

In [56]:
# Restrict to Week 3 feature set for valid manual-vs-pipeline comparison
features = ['Tenure in Months', 'Monthly Charge', 'Contract', 'Payment Method', 'Internet Type', 'Total Charges']
target = 'Churn Label'

X = df[features].copy()
y = df[target].map({'Yes': 1, 'No': 0})

numerical_cols = ['Tenure in Months', 'Monthly Charge', 'Total Charges']
categorical_cols = ['Contract', 'Payment Method', 'Internet Type']

print(X.dtypes)
print()
print(X.isnull().sum())

Tenure in Months      int64
Monthly Charge      float64
Contract                str
Payment Method          str
Internet Type           str
Total Charges       float64
dtype: object

Tenure in Months       0
Monthly Charge         0
Contract               0
Payment Method         0
Internet Type       1526
Total Charges          0
dtype: int64


## 8. Train/Test Split & Preprocessing Pipeline

Building a `ColumnTransformer`: `StandardScaler` for numerical columns, `SimpleImputer` + `OneHotEncoder` for categorical columns.

In [57]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Preprocessing pipeline for categorical columns: impute missing -> one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='No Internet Service')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Preprocessing for numerical columns: scale
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Combine into a single ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

print(preprocessor)

Train shape: (5634, 6)
Test shape: (1409, 6)
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['Tenure in Months', 'Monthly Charge',
                                  'Total Charges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='No '
                                                                           'Internet '
                                                                           'Service',
                                                                strategy='constant')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Contract', 'Payment Method',
                                  'Internet Type']

## 9. Full Pipeline: Preprocessing + Model

Chaining the preprocessor with Logistic Regression and Decision Tree classifiers.

In [58]:
# Full pipeline: preprocessing + Logistic Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Full pipeline: preprocessing + Decision Tree
tree_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# Fit both
lr_pipeline.fit(X_train, y_train)
tree_pipeline.fit(X_train, y_train)

# Predict
lr_preds = lr_pipeline.predict(X_test)
tree_preds = tree_pipeline.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_preds))
print("Decision Tree Accuracy:", accuracy_score(y_test, tree_preds))

Logistic Regression Accuracy: 0.801277501774308
Decision Tree Accuracy: 0.7423704755145494


## 10. Model Evaluation & Comparison with Week 3

| Metric | Week 3 (Manual) | Week 4 (Pipeline) |
|---|---|---|
| Logistic Regression Accuracy | 80.3% | 80.13% |
| Decision Tree Accuracy | 74.0% | 74.24% |

Results closely match the manual approach, confirming the pipeline is correct.

In [59]:
print("=== Logistic Regression ===")
print(classification_report(y_test, lr_preds, target_names=['No Churn', 'Churn']))

print("=== Decision Tree ===")
print(classification_report(y_test, tree_preds, target_names=['No Churn', 'Churn']))

=== Logistic Regression ===
              precision    recall  f1-score   support

    No Churn       0.85      0.89      0.87      1035
       Churn       0.65      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409

=== Decision Tree ===
              precision    recall  f1-score   support

    No Churn       0.83      0.82      0.82      1035
       Churn       0.51      0.52      0.52       374

    accuracy                           0.74      1409
   macro avg       0.67      0.67      0.67      1409
weighted avg       0.74      0.74      0.74      1409



## 11. Feature Engineering

Adding two new features:
- `Charge_per_Tenure` = Monthly Charge / (Tenure + 1) — spending intensity relative to how long the customer has stayed
- `Has_Internet` = binary flag for whether the customer has internet service

In [60]:
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

for df_ in [X_train_fe, X_test_fe]:
    df_['Charge_per_Tenure'] = df_['Monthly Charge'] / (df_['Tenure in Months'] + 1)
    df_['Has_Internet'] = df_['Internet Type'].notna().astype(int)

print(X_train_fe[['Monthly Charge', 'Tenure in Months', 'Charge_per_Tenure', 'Internet Type', 'Has_Internet']].head())

      Monthly Charge  Tenure in Months  Charge_per_Tenure Internet Type  \
4626            55.3                16           3.252941           DSL   
4192           105.3                12           8.100000   Fiber Optic   
5457            49.0                 1          24.500000           DSL   
4717            68.4                58           1.159322         Cable   
4673            80.0                 3          20.000000   Fiber Optic   

      Has_Internet  
4626             1  
4192             1  
5457             1  
4717             1  
4673             1  


## 12. Performance With Engineered Features

| Model | Without Engineered Features | With Engineered Features |
|---|---|---|
| Logistic Regression | 80.13% | 80.34% |
| Decision Tree | 74.24% | 74.66% |

In [61]:
print(X_train_fe['Has_Internet'].value_counts())

# Updated column lists including engineered features
numerical_cols_fe = numerical_cols + ['Charge_per_Tenure']
categorical_cols_fe = categorical_cols  # Has_Internet is already binary 0/1, treat as numerical
numerical_cols_fe = numerical_cols_fe + ['Has_Internet']

preprocessor_fe = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols_fe),
    ('cat', categorical_transformer, categorical_cols_fe)
])

lr_pipeline_fe = Pipeline(steps=[
    ('preprocessor', preprocessor_fe),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

tree_pipeline_fe = Pipeline(steps=[
    ('preprocessor', preprocessor_fe),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

lr_pipeline_fe.fit(X_train_fe, y_train)
tree_pipeline_fe.fit(X_train_fe, y_train)

lr_preds_fe = lr_pipeline_fe.predict(X_test_fe)
tree_preds_fe = tree_pipeline_fe.predict(X_test_fe)

print("Logistic Regression Accuracy (with engineered features):", accuracy_score(y_test, lr_preds_fe))
print("Decision Tree Accuracy (with engineered features):", accuracy_score(y_test, tree_preds_fe))

Has_Internet
1    4417
0    1217
Name: count, dtype: int64
Logistic Regression Accuracy (with engineered features): 0.8034066713981547
Decision Tree Accuracy (with engineered features): 0.7466288147622427


## 13. Saving the Final Pipeline

In [62]:
joblib.dump(lr_pipeline_fe, 'churn_pipeline_logistic.joblib')
print("Pipeline saved successfully.")

# Verify by loading back
loaded_pipeline = joblib.load('churn_pipeline_logistic.joblib')
test_preds = loaded_pipeline.predict(X_test_fe)
print("Loaded pipeline accuracy check:", accuracy_score(y_test, test_preds))

Pipeline saved successfully.


Loaded pipeline accuracy check: 0.8034066713981547


## 14. Conclusion

Built a single reusable `Pipeline` combining a `ColumnTransformer` (StandardScaler + SimpleImputer/OneHotEncoder) with a classifier, replacing manual preprocessing from Week 3. Verified the pipeline reproduces Week 3's manual results (Logistic Regression: 80.13% vs 80.3%). Added two engineered features (`Charge_per_Tenure`, `Has_Internet`) which improved both models slightly. Final pipeline saved with `joblib` and verified by reloading.